## NeuroOnco Immune 

#### Section 1: API Data Harvesting

In [2]:
import requests
import pandas as pd
import numpy as np

def fetch_chembl_egfr_activity(target_id="CHEMBL203", limit=1000):
    base_url = "https://www.ebi.ac.uk/chembl/api/data/activity.json"
    params = {
        "target_chembl_id" : target_id,
        "standard_type" : "IC50",
        "limit" : limit
    }
    try:
        response = requests.get(base_url, params=params)
    except requests.RequestException as e:
        raise RuntimeError(f"ChEMBL request failed: {e}")
    
    if response.status_code == 200:
        data = response.json()
        print(type(data))
        print(data.keys())
        df_bio = pd.DataFrame(data["activities"])
        return df_bio
    else:
        raise RuntimeError(f"ChEMBL API request failed. Status code: {response.status_code}")

def fetch_cbioportal_gbm(study_id="gbm_tcga_pub"):
    attributes_url = f"https://www.cbioportal.org/api/studies/{study_id}/clinical-attributes"
    try:
        response = requests.get(attributes_url)
    except requests.RequestException as e:
        raise RuntimeError(f"cBioPortal API request failed. Status code: {e}")

    if response.status_code == 200:
        attributes_data = response.json()
        print(type(attributes_data))
        print(attributes_data[0])
        df_attributes = pd.DataFrame(attributes_data)
        print("Available attributes: ", df_attributes["clinicalAttributeId"].unique())
    else:
        raise RuntimeError(f"cBioPortal API request failed. Status code: {response.status_code}")
    data_url = f"https://www.cbioportal.org/api/studies/{study_id}/clinical-data"
    params = {
        "clinicalDataType" : "PATIENT"
    }
    try: 
        response = requests.get(data_url, params=params)
    except requests.RequestException as e:
        raise RuntimeError(f"cBioPortal request failed; {e}")
    print("Request URL: ", response.url)
    if response.status_code == 200:
        data = response.json()
        print(f"\nFirst clinical data record: ", data[0])
        df_cbio = pd.DataFrame(data)
        print("\nClinical-data columns:", df_cbio.columns.tolist())
        print("\nClinical attributes actually returned:", df_cbio["clinicalAttributeId"].unique())
        desired_attributes = [
            "SEX", 
            "OS_MONTHS",
            "OS_STATUS",
        ]
        df_cbio = df_cbio[df_cbio["clinicalAttributeId"].isin(desired_attributes)]
        print("Rows after filtering", df_cbio.shape)
        if df_cbio.empty:
            raise RuntimeError("No SEX, OS_MONTHS, or OS_STATUS records were returned by the clinical-data endpoint.")
        df_clinical = df_cbio.pivot(
            index="patientId",
            columns="clinicalAttributeId",
            values="value"
        ).reset_index()
        return df_clinical
    else:
        raise RuntimeError(
            f"cBioPortal clinical-data request failed. "
            f"Status code: {response.status_code}\n"
            f"{response.text}"
        )

df_chembl_raw = fetch_chembl_egfr_activity()
df_clinical_raw = fetch_cbioportal_gbm()
print("\nChEMBL", df_chembl_raw.columns.tolist())
print("\ncBioPortal", df_clinical_raw.columns.tolist())
print("ChEMBL head: ")
print(df_chembl_raw[["molecule_chembl_id", "standard_value", "standard_units", "pchembl_value"]].head())
print("cBioPortal GBM Clinical head: ")
print(df_clinical_raw[["patientId",  "SEX", "OS_MONTHS", "OS_STATUS"]].head())


<class 'dict'>
dict_keys(['activities', 'page_meta'])
<class 'list'>
{'displayName': 'ACGH Data', 'description': 'ACGH Data.', 'datatype': 'STRING', 'patientAttribute': False, 'priority': '1', 'clinicalAttributeId': 'ACGH_DATA', 'studyId': 'gbm_tcga_pub'}
Available attributes:  ['ACGH_DATA' 'CANCER_TYPE' 'CANCER_TYPE_DETAILED' 'COMPLETE_DATA'
 'DFS_MONTHS' 'DFS_STATUS' 'FRACTION_GENOME_ALTERED' 'ICD_10'
 'KARNOFSKY_PERFORMANCE_SCORE' 'MRNA_DATA' 'MUTATION_COUNT'
 'ONCOTREE_CODE' 'OS_MONTHS' 'OS_STATUS' 'PRETREATMENT_HISTORY'
 'PRIOR_GLIOMA' 'SAMPLE_COUNT' 'SAMPLE_TYPE' 'SEQUENCED' 'SEX'
 'SOMATIC_STATUS' 'TMB_NONSYNONYMOUS' 'TREATMENT_STATUS']
Request URL:  https://www.cbioportal.org/api/studies/gbm_tcga_pub/clinical-data?clinicalDataType=PATIENT

First clinical data record:  {'uniquePatientKey': 'VENHQS0wMi0wMDAxOmdibV90Y2dhX3B1Yg', 'patientId': 'TCGA-02-0001', 'studyId': 'gbm_tcga_pub', 'clinicalAttributeId': 'DFS_MONTHS', 'value': '4.504109589'}

Clinical-data columns: ['uniquePatie

#### Data Cleaning & Parsing

In [12]:
def clean_chembl_bioactivity(df_chembl_raw) -> pd.DataFrame:

    df = df_chembl_raw.copy() # good practice!
    df = df_chembl_raw.dropna(subset=["standard_value"])

    df["standard_value"] = pd.to_numeric(df["standard_value"], errors="coerce")

    if "pchembl_value" in df.columns:
        df["pchembl_value"] = pd.to_numeric(df["pchembl_value"], errors="coerce")
    else:
        df["pchembl_value"] = np.nan

    if "standard_units" in df.columns: 
        df = df[df["standard_units"].astype(str).str.upper() == "NM"]

    missing_mask = df["pchembl_value"].isna() & df["standard_value"].notna()
    molar_vals =  df.loc[missing_mask, "standard_value"] * 1e-9
    df.loc[missing_mask, "pchembl_value"] = -np.log10(molar_vals)

    df = df.dropna(subset=["pchembl_value"])
    df = df.sort_values(by="pchembl_value", ascending=False)
    df= df.drop_duplicates(subset=["molecule_chembl_id"], keep="first").reset_index(drop=True)

    return df

def clean_cbioportal_clinical(df_clinical_raw) -> pd.DataFrame:

    df = df_clinical_raw.copy()
    numeric_candidates = [
        "OS_MONTHS",
        "DFS_MONTHS",
        "KARNOFSKY_PERFORMANCE_SCORE",
        "SAMPLE_COUNT"
    ]
    for col in numeric_candidates:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    if "OS_STATUS" in df.columns:
        df["OS_STATUS_CLEAN"] = df["OS_STATUS"].astype(str).str.split(":").str[-1].str.strip()

    if "OS_MONTHS" in df.columns:
        df = df.dropna(subset=["OS_MONTHS"]).reset_index(drop=True)

    return df

df_chembl_clean = clean_chembl_bioactivity(df_chembl_raw)
df_clinical_clean = clean_cbioportal_clinical(df_clinical_raw)

df_chembl_clean.to_csv("data/chembl_egfr_bioactivity_clean.csv", index=False)
df_clinical_clean.to_csv("data/tcga_gbm_clinical_clean.csv", index=False)

print("\n Cleaning Summary")
print("ChEMBL DF Shape: ", df_chembl_clean.shape)
print("Clinical Features: ", df_clinical_clean.columns.tolist())


 Cleaning Summary
ChEMBL DF Shape:  (780, 47)
Clinical Features:  ['patientId', 'OS_MONTHS', 'OS_STATUS', 'SEX', 'OS_STATUS_CLEAN']


C:\Users\Korisnik\AppData\Local\Temp\ipykernel_6152\2217664238.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["standard_value"] = pd.to_numeric(df["standard_value"], errors="coerce")
C:\Users\Korisnik\AppData\Local\Temp\ipykernel_6152\2217664238.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["pchembl_value"] = pd.to_numeric(df["pchembl_value"], errors="coerce")


#### Statistical Dispersion

In [ ]:
def analyze_chembl_dispersion(df_chembl) -> dict:
    pchembl = df_chembl["pchembl_value"].dropna()
    pchembl_range = pchembl.max() - pchembl.min()
    pchembl_std = pchembl.std()

    q25 = pchembl.quantile(0.25)
    q75 = pchembl.quantile(0.75)
    iqr = q75 - q25

    lower_bound = q25 - (1.5 * iqr)
    upper_bound = q75 - (1.5 * iqr)
    outliers = df_chembl[
        (df_chembl["pchembl_value"] < lower_bound) |
        (df_chembl["pchembl_value"] > upper_bound)
    ]

    metrics = {
        "count" : len(pchembl),
        "mean" : pchembl.mean(),
        "median" : pchembl.median(),
        "range" : pchembl_range,
        "iqr" : iqr,
        "std" : pchembl_std,
        "num_outliers" : len(outliers)
    }

    print("ChEMBL EGFR Bioactivity Dispersion")
    for k, v in metrics.items():
        print (f"{k} : {v:.3f}" if isinstance(v, float) else f"{k} : {v}")

    return metrics

def analyze_clinical_dispersion(df_clinical) -> pd.DataFrame:
    def compute_iqr(x):
        return x.quantile(0.75) - x.quantile(0.25)

    stats_df = (
        df_clinical
        .groupby("OS_STATUS_CLEAN")["OS_MONTHS"]
        .agg(["count", "mean", "median", "std", compute_iqr])
        .rename(columns={"compute_iqr" : "iqr"})
    )

    print("cBioPortal Glioblastoma Survival by Status")
    print(stats_df)

    return stats_df

chembl_stats = analyze_chembl_dispersion(df_chembl_clean)
clinical_stats = analyze_clinical_dispersion(df_clinical_clean)

SyntaxError: unmatched ')' (2897895270.py, line 3)